# Impulse — A2D2 Object Detection → Time-Series Channels

Third notebook of the A2D2 demo. It runs a **pretrained torchvision COCO object detector
(BSD-3-Clause)** over the camera frames already extracted to the Unity Catalog volume by
`a2d2_ingestion.ipynb`, **distributed across Spark workers** (`mapInPandas`), turns per-frame
detection counts into **numerical time-series channels** (e.g. `detected_cyclists` = number of
bicycles seen at each frame timestamp), and **appends** them to the silver layer
(`channels` / `channel_tags` / `channel_metrics`) so they become first-class TSAL channels
alongside the bus signals.

## Prerequisite
Run `a2d2_ingestion.ipynb` first (with `download_images=true`). This notebook reads
`{catalog}.{schema}.{table_prefix}_camera_frames` and the PNGs on the volume; it downloads
nothing except the model weights (cached to the volume). torchvision/COCO weights are BSD-3;
the A2D2 data and weights stay on your own volume — commit the notebook with outputs cleared.

In [ ]:
%pip install torch torchvision pillow -q
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume")
dbutils.widgets.dropdown("detect_model", "fasterrcnn_resnet50_fpn",
    ["fasterrcnn_resnet50_fpn", "ssdlite320_mobilenet_v3_large", "fcos_resnet50_fpn"], "Detection model")
dbutils.widgets.text("detect_score_threshold", "0.5", "Score threshold")
dbutils.widgets.text("detect_repartitions", "16", "Inference partitions")

In [ ]:
import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
MODEL_NAME = dbutils.widgets.get("detect_model")
SCORE_THRESHOLD = float(dbutils.widgets.get("detect_score_threshold") or "0.5")
REPARTITIONS = int(dbutils.widgets.get("detect_repartitions") or "16")

if not (CATALOG and SCHEMA and TABLE_PREFIX and VOLUME):
    raise ValueError("Please set catalog, schema, table_prefix and volume widgets above.")

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-3])
src_dir = os.path.join(REPO_ROOT, "src")
if os.path.isdir(src_dir):
    sys.path.insert(0, src_dir)

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
CONTAINER_ID = 1
vol_root = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Curated COCO classes -> channels (value = per-frame count). Name-keyed so it survives
# COCO's 91-index gaps; resolved to indices on the driver below.
CLASS_MAP = {
    "detected_pedestrians":    {"person"},
    "detected_cyclists":       {"bicycle"},
    "detected_motorcycles":    {"motorcycle"},
    "detected_cars":           {"car"},
    "detected_buses":          {"bus"},
    "detected_trucks":         {"truck"},
    "detected_vehicles":       {"car", "bus", "truck", "motorcycle"},
    "detected_traffic_lights": {"traffic light"},
    "detected_stop_signs":     {"stop sign"},
}
DETECTED_CHANNELS = list(CLASS_MAP.keys())
DETECT_ID_BASE = 100  # detection channel_ids 100.. (never collide with bus ids 1..22)
channel_id_for = {name: DETECT_ID_BASE + i for i, name in enumerate(DETECTED_CHANNELS)}
print(f"model={MODEL_NAME} thr={SCORE_THRESHOLD} | channels: {DETECTED_CHANNELS}")

## Target classes → channels

| channel | COCO class(es) |
|---|---|
| `detected_pedestrians` | person |
| `detected_cyclists` | bicycle |
| `detected_motorcycles` | motorcycle |
| `detected_cars` | car |
| `detected_buses` | bus |
| `detected_trucks` | truck |
| `detected_vehicles` | car + bus + truck + motorcycle (aggregate) |
| `detected_traffic_lights` | traffic light |
| `detected_stop_signs` | stop sign |

In [ ]:
frames_df = (spark.read.table(f"{pfx}_camera_frames")
             .select("container_id", "frame_id", "cam_tstamp_us", "png_path")
             .where("png_path IS NOT NULL"))
n_frames = frames_df.count()
if n_frames == 0:
    raise RuntimeError(f"{pfx}_camera_frames is empty - run a2d2_ingestion with download_images=true first.")
print(f"{n_frames} camera frames to score with {MODEL_NAME}")

In [ ]:
import torch
from torchvision.models import detection as D

# model name -> (constructor, weights enum)
MODELS = {
    "fasterrcnn_resnet50_fpn":       (D.fasterrcnn_resnet50_fpn,       D.FasterRCNN_ResNet50_FPN_Weights),
    "ssdlite320_mobilenet_v3_large": (D.ssdlite320_mobilenet_v3_large, D.SSDLite320_MobileNet_V3_Large_Weights),
    "fcos_resnet50_fpn":             (D.fcos_resnet50_fpn,             D.FCOS_ResNet50_FPN_Weights),
}
ctor, wenum = MODELS[MODEL_NAME]
weights = wenum.DEFAULT
COCO_CATEGORIES = list(weights.meta["categories"])  # index -> class name (bundled, no download)

# Download weights ONCE on the driver and cache the state_dict on the volume; workers then
# load it from the FUSE path (no worker internet egress needed).
models_dir = f"{vol_root}/models"
os.makedirs(models_dir, exist_ok=True)
weights_path = f"{models_dir}/{MODEL_NAME}.pth"
if not os.path.exists(weights_path):
    print("downloading weights on driver ...")
    _m = ctor(weights=weights)
    torch.save(_m.state_dict(), weights_path)
    del _m
print("weights cached at:", weights_path)

# Resolve channel -> set of COCO indices on the driver. These small values are captured
# directly by the mapInPandas closure (serverless/Spark Connect forbids
# spark.sparkContext.broadcast; the big weights file is read by workers from the volume).
name_to_idx = {c: i for i, c in enumerate(COCO_CATEGORIES)}
CLASS_IDX = {ch: sorted({name_to_idx[n] for n in names if n in name_to_idx})
             for ch, names in CLASS_MAP.items()}
print("class->COCO indices:", CLASS_IDX)

## Distributed inference (`mapInPandas`)

Each Spark partition loads the model once (from the cached weights, built with
`weights=None` so workers never download), scores its frames on CPU, filters by score
threshold, and emits one long-form row per (frame, channel) with the detection count.
`detect_repartitions` controls parallelism.

In [ ]:
import pyspark.sql.types as T

DETECT_OUT_SCHEMA = T.StructType([
    T.StructField("container_id",  T.LongType(),    False),
    T.StructField("cam_tstamp_us", T.LongType(),    False),
    T.StructField("channel_name",  T.StringType(),  False),
    T.StructField("count",         T.IntegerType(), False),
])

def detect_partition(iterator):
    import torch
    from torchvision.models import detection as _D
    import torchvision.transforms.functional as _TF
    from PIL import Image
    from collections import Counter
    import pandas as pd

    ctors = {
        "fasterrcnn_resnet50_fpn":       _D.fasterrcnn_resnet50_fpn,
        "ssdlite320_mobilenet_v3_large": _D.ssdlite320_mobilenet_v3_large,
        "fcos_resnet50_fpn":             _D.fcos_resnet50_fpn,
    }
    # Load the model once per partition (architecture only -> NO download; weights from FUSE).
    # weights_path / MODEL_NAME / SCORE_THRESHOLD / CLASS_IDX are captured from the notebook
    # globals by the closure (no broadcast — serverless forbids spark.sparkContext).
    model = ctors[MODEL_NAME](weights=None, weights_backbone=None)
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    torch.set_num_threads(1)
    thr = SCORE_THRESHOLD
    class_idx = CLASS_IDX

    for pdf in iterator:
        rows = []
        for _, r in pdf.iterrows():
            try:
                img = Image.open(r["png_path"]).convert("RGB")
            except Exception:
                continue
            x = _TF.to_tensor(img)
            with torch.no_grad():
                pred = model([x])[0]
            keep = pred["scores"] >= thr
            cnt = Counter(pred["labels"][keep].tolist())
            for ch, idxs in class_idx.items():
                c = int(sum(cnt.get(i, 0) for i in idxs))
                rows.append((int(r["container_id"]), int(r["cam_tstamp_us"]), ch, c))
        if rows:
            yield pd.DataFrame(rows, columns=["container_id", "cam_tstamp_us", "channel_name", "count"])

det_pdf = (frames_df.repartition(REPARTITIONS)
           .mapInPandas(detect_partition, schema=DETECT_OUT_SCHEMA)
           .toPandas())
print(f"detection rows: {len(det_pdf):,} ({len(DETECTED_CHANNELS)} channels x {n_frames} frames)")

## Build silver channels and append

Each `detected_*` channel becomes RAW points (timestamp = frame `cam_tstamp_us`, value =
count), with `channel_tags` (channel_name / unit=count / source) and `channel_metrics`
(same `SampleSeries` recipe as ingestion). Append is idempotent: delete this run's
channel_ids first, then append — so it survives the ingest overwrite and standalone re-runs.

In [ ]:
import numpy as np, pandas as pd
from impulse_query_engine import schema as S
from impulse_query_engine.model.series.sample_series import SampleSeries

def weighted_std(values, w):
    wsum = w.sum()
    if wsum <= 0:
        return float("nan")
    m = np.nansum(values * w) / wsum
    return float(np.sqrt(np.nansum(w * (values - m) ** 2) / wsum))

def weighted_quantiles(values, w, qs):
    order = np.argsort(values)
    v, ww = values[order], w[order]
    cw = np.cumsum(ww)
    if cw[-1] <= 0:
        return [float("nan")] * len(qs)
    cw = cw / cw[-1]
    return [float(np.interp(q, cw, v)) for q in qs]

ch_parts, metric_rows, tag_rows = [], [], []
for name in DETECTED_CHANNELS:
    cid = channel_id_for[name]
    sub = det_pdf[det_pdf["channel_name"] == name].sort_values("cam_tstamp_us")
    times = sub["cam_tstamp_us"].to_numpy(np.float64)
    values = sub["count"].to_numpy(np.float64)
    n = int(len(values))
    ch_parts.append(pd.DataFrame({
        "container_id": np.int64(CONTAINER_ID),
        "channel_id": np.int32(cid),
        "timestamp": sub["cam_tstamp_us"].to_numpy(np.int64),
        "value": values,
    }))
    tag_rows += [
        (CONTAINER_ID, cid, "channel_name", name),
        (CONTAINER_ID, cid, "unit", "count"),
        (CONTAINER_ID, cid, "source", f"torchvision:{MODEL_NAME}"),
    ]
    if n >= 2:
        ss = SampleSeries.from_timestamps(times / 1e6, values)
        dur = ss.durations()
        pz1, pz10, pz90, pz99 = weighted_quantiles(values, dur, [0.01, 0.10, 0.90, 0.99])
        metric_rows.append((
            CONTAINER_ID, cid, "DOUBLE", n, float(ss.nan_ratio()),
            float(times[0] / 1e6), float(times[-1] / 1e6), int((times[-1] - times[0]) / 1e3),
            n, float(np.mean(np.diff(times)) / 1e6),
            float(ss.min()), float(ss.max()), float(ss.mean()), weighted_std(values, dur),
            pz1, pz10, pz90, pz99,
        ))
    else:
        b = float(times[0] / 1e6) if n else 0.0
        v0 = float(values[0]) if n else float("nan")
        metric_rows.append((CONTAINER_ID, cid, "DOUBLE", n, 0.0, b, b, 0, n, 0.0,
                            v0, v0, v0, float("nan"), v0, v0, v0, v0))

channels_df = spark.createDataFrame(pd.concat(ch_parts, ignore_index=True), schema=S.CHANNELS_SCHEMA_WITHOUT_RLE)
channel_tags_df = spark.createDataFrame(tag_rows, schema=S.CHANNEL_TAGS)
channel_metrics_df = spark.createDataFrame(metric_rows, schema=S.CHANNEL_METRICS)

id_list = ",".join(str(channel_id_for[n]) for n in DETECTED_CHANNELS)
for tbl, df in [("channels", channels_df), ("channel_tags", channel_tags_df), ("channel_metrics", channel_metrics_df)]:
    full = f"{pfx}_{tbl}"
    spark.sql(f"DELETE FROM {full} WHERE container_id = {CONTAINER_ID} AND channel_id IN ({id_list})")
    df.write.mode("append").saveAsTable(full)

# keep container_metrics.num_channels in sync with the distinct channel count
total = (spark.read.table(f"{pfx}_channel_tags").where("key = 'channel_name'")
         .select("channel_id").distinct().count())
spark.sql(f"UPDATE {pfx}_container_metrics SET num_channels = {total} WHERE container_id = {CONTAINER_ID}")
print(f"Appended {len(DETECTED_CHANNELS)} detected_* channels (ids {id_list}); total channels now {total}.")

## Validation — detected channels are first-class TSAL channels

In [ ]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
import pyspark.sql.functions as F

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channel_tags_table": f"{pfx}_channel_tags",
        "channels_uri": f"{pfx}_channels",
    },
    "unity_sink": {"catalog": CATALOG, "schema": SCHEMA, "table_prefix": TABLE_PREFIX},
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id"],
}
report = Report(name="a2d2_detect", spark=spark, workspace_client=WorkspaceClient(), config=config)
db = report.get_db()
_ = db.query.channel(channel_name="detected_pedestrians")  # resolves => first-class channel
print("OK: detected_pedestrians resolves as a TSAL channel.")

print("Total detections per channel:")
display(spark.createDataFrame(det_pdf)
        .groupBy("channel_name").agg(F.sum("count").alias("total_detections"))
        .orderBy(F.desc("total_detections")))
display(spark.read.table(f"{pfx}_channel_metrics").where(f"channel_id IN ({id_list})")
        .select("channel_id", "sample_count", "min", "max", "mean"))